# SETUP

In [1]:
# Janky code to do different setup when run in a Colab notebook vs VSCode
import os

from IPython import get_ipython

ipython = get_ipython()
# Code to automatically update the HookedTransformer code as its edited without restarting the kernel
ipython.magic("load_ext autoreload")
ipython.magic("autoreload 2")

# Import stuff
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import einops
from fancy_einsum import einsum
import tqdm.notebook as tqdm
import random
from pathlib import Path
import plotly.express as px
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt

from torchtyping import TensorType as TT
from typing import List, Union, Optional, Callable
from functools import partial
import copy
import itertools
import json

from transformers import AutoModelForCausalLM, AutoConfig, AutoTokenizer
import dataclasses
import datasets
from IPython.display import HTML, Markdown
import umap

/var/folders/jq/3qm60jc56v336bp7cy96fxgm0000gn/T/ipykernel_14591/2315760820.py:8: DeprecationWarning: `magic(...)` is deprecated since IPython 0.13 (warning added in 8.1), use run_line_magic(magic_name, parameter_s).
  ipython.magic("load_ext autoreload")
/var/folders/jq/3qm60jc56v336bp7cy96fxgm0000gn/T/ipykernel_14591/2315760820.py:9: DeprecationWarning: `magic(...)` is deprecated since IPython 0.13 (warning added in 8.1), use run_line_magic(magic_name, parameter_s).
  ipython.magic("autoreload 2")
/opt/homebrew/Caskroom/miniconda/base/envs/atrb-patching/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import transformer_lens
import transformer_lens.utils as utils
from transformer_lens.hook_points import (
    HookedRootModule,
    HookPoint,
)  # Hooking utilities
from transformer_lens import (
    HookedTransformer,
    HookedTransformerConfig,
    FactoredMatrix,
    ActivationCache,
)

In [3]:
from neel_plotly import line, imshow, scatter
import plotly.io as pio
pio.renderers.default = 'vscode'

from tqdm import tqdm

import transformer_lens.patching as patching

from typing_extensions import Literal

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from sklearn.decomposition import PCA

import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker

## IOI SETUP

In [4]:
model = HookedTransformer.from_pretrained("gpt2-small")


model.set_use_attn_result(True)
model.set_use_attn_in(True)
model.set_use_hook_mlp_in(True)

Loaded pretrained model gpt2-small into HookedTransformer


In [5]:
import random

names = [
    "John", "Mary", "Tom", "James", "Dan", "Sid", "Martin", "Amy", "Alice", "Bob",
    "Charlie", "Emma", "David", "Sarah", "Michael", "Laura", "Kevin", "Olivia", 
    "Nathan", "Chloe", "Ethan", "Lucy", "Ryan", "Zoe", "Alex", "Bella", "Jack", 
    "Sophie", "Liam", "Emily"
]

objects = [
    "bag", "ball", "apple", "drink", "book", "toy", "pen", "hat", "chair", "phone", 
    "watch", "candle", "laptop", "pillow", "guitar", "notebook", "camera", "wallet", 
    "shirt", "jacket", "basket", "spoon", "mirror", "umbrella", "ring", "necklace", 
    "glasses", "bottle", "scarf", "brush"
]

actions = [
    "gave", "handed", "passed", "offered", "tossed", "delivered", "presented", 
    "supplied", "lent", "donated", "brought", "gifted", "transferred", "conveyed", 
    "shared"
]

places = [
    "shops", "park", "library", "beach", "school", "mall", "cafe", "theater", 
    "museum", "garden", "stadium", "market", "farm", "fair", "pier", "station", 
    "zoo", "restaurant", "gym", "airport"
]

times = ["When", "After"]

def generate_sentences(num=20):
    sentences = []
    answers = []
    for _ in range(num):
        name1, name2 = random.sample(names, 2)
        obj = random.choice(objects)
        action = random.choice(actions)
        place = random.choice(places)
        time = random.choice(times)
        
        sentences.append(f"{time} {name1} and {name2} went to the {place}, {name1} {action} the {obj} to")
        sentences.append(f"{time} {name1} and {name2} went to the {place}, {name2} {action} the {obj} to")
        
        answers.append((" " + name1, " " + name2))
        answers.append((" " + name2, " " + name1))
    
    return sentences, answers



In [8]:
def get_logit_diff(logits, answer_token_indices):
    if len(logits.shape) == 3:
        # Get final logits only
        logits = logits[:, -1, :]
    correct_logits = logits.gather(1, answer_token_indices[:, 0].unsqueeze(1))
    incorrect_logits = logits.gather(1, answer_token_indices[:, 1].unsqueeze(1))
    return (correct_logits - incorrect_logits).mean()


# clean_logits, clean_cache = model.run_with_cache(clean_tokens)
# corrupted_logits, corrupted_cache = model.run_with_cache(corrupted_tokens)

# clean_logit_diff = get_logit_diff(clean_logits, answer_token_indices).item()
# print(f"Clean logit diff: {clean_logit_diff:.4f}")

# corrupted_logit_diff = get_logit_diff(corrupted_logits, answer_token_indices).item()
# print(f"Corrupted logit diff: {corrupted_logit_diff:.4f}")

In [ ]:
CLEAN_BASELINE = clean_logit_diff
CORRUPTED_BASELINE = corrupted_logit_diff


def ioi_metric(logits, answer_token_indices=answer_token_indices):
    return (get_logit_diff(logits, answer_token_indices) - CORRUPTED_BASELINE) / (
        CLEAN_BASELINE - CORRUPTED_BASELINE
    )

In [10]:
def filter_mlp_hooks(name):
    return "mlp" in name

def get_cache_fwd_and_bwd(model, tokens, metric):
    model.reset_hooks()
    cache = {}

    def forward_cache_hook(act, hook):
        cache[hook.name] = act.detach()

    model.add_hook(filter_mlp_hooks, forward_cache_hook, "fwd")

    grad_cache = {}

    def backward_cache_hook(act, hook):
        grad_cache[hook.name] = act.detach()

    model.add_hook(filter_mlp_hooks, backward_cache_hook, "bwd")

    value = metric(model(tokens))
    value.backward()
    model.reset_hooks()
    return (
        value.item(),
        ActivationCache(cache, model),
        ActivationCache(grad_cache, model),
    )



In [11]:
def stack_mlp_from_cache(cache, activation_name: Literal["mlp_in", "mlp_out", "pre", "post"]):
    stacked_mlp_vectors = torch.stack(
        [cache[activation_name, l] for l in range(model.cfg.n_layers)], dim=0
    )

    return stacked_mlp_vectors

# Get activation vector for large batch size

# Clustering

 - For each neuron we will compute an attribuition score per sample vector.
 - Denote $attr_{x_i}$ the attribution score for neuron $x$ on input $i$.
 - For each neuron $x$ we will compute vector $v_x \in \R^{batch}$ where $v_{x_i} = atr_{x_i}$

In [ ]:
num_layers = model.cfg.n_layers

def get_sample_attribution_vectors(clean_cache, sum_over_pos=False):
    clean_act = stack_mlp_from_cache(clean_cache, "post")
    corrupted_act = stack_mlp_from_cache(corrupted_cache, "post")
    grad_vector = stack_mlp_from_cache(corrupted_grad_cache, "post")
    
    attribution_scores = einops.rearrange(
        grad_vector * (clean_act - corrupted_act),
        "layer batch pos dim -> (layer pos dim) batch"
    )
    
    return attribution_scores

In [12]:
num_layers = model.cfg.n_layers

def get_sample_activation_vector(clean_cache, sum_over_pos=False):
    clean_act = stack_mlp_from_cache(clean_cache, "post")

    attribution_scores = einops.rearrange(
        clean_act,
        "layer batch pos dim -> layer (pos dim) batch"
    )
    
    return attribution_scores

In [ ]:
def throw_low_attribution_neuron(attributions, clean_cache, corrupted_cache, grad_cache, sum_over_pos=False, threshold=0.1):
    clean_act = stack_mlp_from_cache(clean_cache, "post")
    corrupted_act = stack_mlp_from_cache(corrupted_cache, "post")
    grad_vector = stack_mlp_from_cache(grad_cache, "post")
    
    attribution_scores = einops.rearrange(
        grad_vector * (clean_act - corrupted_act),
        "layer batch pos d_mlp -> layer (pos d_mlp) batch",
    )

    mask = attribution_scores > threshold

    while(mask.dim() < attributions.dim()):
        mask = mask.unsqueeze(-1)
    
    
    mask = mask.expand_as(attributions)
    
    filtered_out = attributions * mask
    
    return filtered_out

# Batching

In [17]:
batch_size = 16
batches = 64
total_samples = batch_size * batches

prompts, answers = generate_sentences(int(total_samples/2))

#first batch 
clean_tokens = model.to_tokens(prompts[:batch_size])

corrupted_tokens = clean_tokens[
    [(i + 1 if i % 2 == 0 else i - 1) for i in range(len(clean_tokens))]
]

answer_token_indices = torch.tensor(
    [
        [model.to_single_token(answers[i][j]) for j in range(2)]
        for i in range(batch_size)
    ],
    device=model.cfg.device,
)

clean_logits, clean_cache = model.run_with_cache(clean_tokens)
corrupted_logits, corrupted_cache = model.run_with_cache(corrupted_tokens)

clean_logit_diff = get_logit_diff(clean_logits, answer_token_indices).item()
corrupted_logit_diff = get_logit_diff(corrupted_logits, answer_token_indices).item()

CLEAN_BASELINE = clean_logit_diff
CORRUPTED_BASELINE = corrupted_logit_diff


def ioi_metric(logits, answer_token_indices=answer_token_indices):
    return (get_logit_diff(logits, answer_token_indices) - CORRUPTED_BASELINE) / (
        CLEAN_BASELINE - CORRUPTED_BASELINE
    )
    
clean_value, clean_cache, clean_grad_cache = get_cache_fwd_and_bwd(
    model, clean_tokens, ioi_metric
)

corrupted_value, corrupted_cache, corrupted_grad_cache = get_cache_fwd_and_bwd(
    model, corrupted_tokens, ioi_metric
)

clean_act = stack_mlp_from_cache(clean_cache, "post")
corrupted_act = stack_mlp_from_cache(corrupted_cache, "post")
grad_vector = stack_mlp_from_cache(corrupted_grad_cache, "post")

attribution_scores = einops.rearrange(
    grad_vector * (clean_act - corrupted_act),
    "layer batch pos dim -> layer (pos dim) batch"
)

In [18]:
activations = get_sample_activation_vector(clean_cache)
for b in tqdm(range(batch_size,total_samples,batch_size)):
    clean_tokens = model.to_tokens(prompts[b:b + batch_size])
    
    clean_logits, clean_cache = model.run_with_cache(clean_tokens)
    
    activations = torch.cat((activations, get_sample_activation_vector(clean_cache)), dim=2)
    print(activations.shape)

torch.Size([12, 46080, 32])
torch.Size([12, 46080, 48])
torch.Size([12, 46080, 64])
torch.Size([12, 46080, 80])
torch.Size([12, 46080, 96])
torch.Size([12, 46080, 112])
torch.Size([12, 46080, 128])
torch.Size([12, 46080, 144])
torch.Size([12, 46080, 160])
torch.Size([12, 46080, 176])
torch.Size([12, 46080, 192])
torch.Size([12, 46080, 208])
torch.Size([12, 46080, 224])
torch.Size([12, 46080, 240])
torch.Size([12, 46080, 256])
torch.Size([12, 46080, 272])
torch.Size([12, 46080, 288])
torch.Size([12, 46080, 304])
torch.Size([12, 46080, 320])
torch.Size([12, 46080, 336])
torch.Size([12, 46080, 352])
torch.Size([12, 46080, 368])
torch.Size([12, 46080, 384])
torch.Size([12, 46080, 400])
torch.Size([12, 46080, 416])
torch.Size([12, 46080, 432])
torch.Size([12, 46080, 448])
torch.Size([12, 46080, 464])
torch.Size([12, 46080, 480])
torch.Size([12, 46080, 496])
torch.Size([12, 46080, 512])
torch.Size([12, 46080, 528])
torch.Size([12, 46080, 544])
torch.Size([12, 46080, 560])
torch.Size([12, 460

In [23]:
from sklearn.cluster import HDBSCAN

clusterer = HDBSCAN()

clusters = []

for l in range(num_layers):
    non_empty_mask = activations[l].abs().sum(dim=1).bool()
    filtered_data = activations[l, non_empty_mask, :].cpu().detach().numpy()
        
    cluster = clusterer.fit(filtered_data)
    clusters.append(cluster)

    print(f"Layer {l} has {len(set(cluster.labels_))} clusters")

In [ ]:
activations = get_sample_activation_vector(clean_cache)
activations = throw_low_attribution_neuron(activations, clean_cache, corrupted_cache, corrupted_grad_cache, threshold=1e-13)

## DBSCAN Clustering!

In [ ]:
from sklearn.cluster import DBSCAN

clusterer = DBSCAN(eps=0.6, min_samples=50)

clusters = []

for l in range(num_layers):
    non_empty_mask = activations[l].abs().sum(dim=1).bool()
    filtered_data = activations[l, non_empty_mask, :].cpu().detach().numpy()
    
    print(filtered_data.shape)
    
    cluster = clusterer.fit(filtered_data)
    clusters.append(cluster)
    print(f"Layer {l} has {len(set(cluster_labels))} clusters")

## UMAP 

In [ ]:
embeddings = []
for l in range(num_layers):
    non_empty_mask = activations[l].abs().sum(dim=1).bool()
    filtered_data = activations[l, non_empty_mask, :].cpu().detach().numpy()
    
    reducer = umap.UMAP()
    embeddings.append(reducer.fit_transform(filtered_data))

In [ ]:
for l in range(num_layers):
    unique_labels = set(clusters[l].labels_)
    colors = [plt.cm.Spectral(each) for each in np.linspace(0, 1, len(unique_labels))]
    
    core_samples_mask = np.zeros_like(clusters[l].labels_, dtype=bool)
    core_samples_mask[clusters[l].core_sample_indices_] = True
    
    for k, col in zip(unique_labels, colors):
        if k == -1:
            # Black color for noise.
            col = [0, 0, 0, 1]
        class_member_mask = (clusters[l].labels_ == k)
        # Plot core points with larger markers
        xy = embeddings[l][class_member_mask & core_samples_mask]
        plt.plot(xy[:, 0], xy[:, 1], 'o', markerfacecolor=tuple(col),
                markeredgecolor='k', markersize=14)
        # Plot non-core (border) points with smaller markers
        xy = embeddings[l][class_member_mask & ~core_samples_mask]
        plt.plot(xy[:, 0], xy[:, 1], 'o', markerfacecolor=tuple(col),
                markeredgecolor='k', markersize=6)